# TDS Groq Inference Service

Notebook 3: receives invoice JSON from the backend, sends it to a model through Groq Cloud, and returns only the TDS proposal.

**The backend TDS validator remains the final authority.**

In [1]:
!pip install -q -U openai fastapi uvicorn nest_asyncio pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.9 MB/s eta 0:00:00


## 1. Groq API key

Add `GROQ_API_KEY` to Colab Secrets. Never hardcode or print the complete key.

In [10]:
import os
import json
import time
import datetime
import asyncio
import uvicorn
import threading
import traceback
from typing import Any, Dict, List, Optional
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from openai import OpenAI

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise RuntimeError('GROQ_API_KEY is missing. Add it to Colab Secrets.')

print('Groq API key loaded:', GROQ_API_KEY[:5] + '...' + GROQ_API_KEY[-4:])

# Concurrency lock for safe inference
inference_lock = asyncio.Lock()

# Unique Request ID generator (TDS-YYYYMMDD-HHMMSS-NNNN)
_tds_request_counter = 0
_tds_counter_lock = threading.Lock()

def generate_tds_request_id() -> str:
    global _tds_request_counter
    with _tds_counter_lock:
        _tds_request_counter += 1
        now_str = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        return f"TDS-{now_str}-{_tds_request_counter:04d}"

Groq API key loaded: gsk_d...6HMy


## 2. Groq client

In [3]:
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url='https://api.groq.com/openai/v1',
)

MODEL_NAME = 'openai/gpt-oss-120b'
print('TDS model:', MODEL_NAME)

TDS model: openai/gpt-oss-120b


## 3. TDS output schema

The schema prevents the model from swapping `nature_of_payment`, `tds_provision`, and `tds_section`.

In [4]:
TDS_TABLE_ENUM = ['Table 1(ii)', 'Table 2', 'Table 6', 'Table 6(ii)', 'Table 6(iii)', 'Table 8(ii)']
NATURE_ENUM = ['Purchase of goods', 'Professional services', 'Technical services', 'Contractor services', 'Rent', 'Commission or brokerage', 'Freight or transport', 'Other', 'Unknown']

TDS_SCHEMA = {
    'type':'object', 'additionalProperties':False,
    'properties':{
        'tds_assessment':{
            'type':'object', 'additionalProperties':False,
            'properties':{
                'tds_applicable':{'type':['boolean','null']},
                'nature_of_payment':{'type':['string','null'],'enum':NATURE_ENUM+[None]},
                'tds_provision':{'type':['string','null'],'enum':['Section 393',None]},
                'tds_section':{'type':['string','null'],'enum':TDS_TABLE_ENUM+[None]},
                'tds_rate':{'type':['number','null']},
                'tds_base_amount':{'type':['number','null']},
                'proposed_tds_amount':{'type':['number','null']},
                'tds_needs_review':{'type':'boolean'},
                'tds_reasoning':{'type':['string','null']}
            },
            'required':['tds_applicable','nature_of_payment','tds_provision','tds_section','tds_rate','tds_base_amount','proposed_tds_amount','tds_needs_review','tds_reasoning']
        }
    },
    'required':['tds_assessment']
}

## 4. TDS prompt

In [5]:
SYSTEM_PROMPT = '''You are the TDS proposal model in an Indian Accounts Payable system.

You receive an already-extracted invoice JSON. Your ONLY job is to propose the TDS assessment required by this project.

PROJECT FRAMEWORK — CRITICAL:
- This project uses Section 393 and its configured Section 393 tables.
- When a provision is identified, tds_provision MUST be exactly "Section 393".
- tds_section MUST be one of: Table 1(ii), Table 2, Table 6, Table 6(ii), Table 6(iii), Table 8(ii).
- NEVER substitute another provision such as Section 194Q, 194C, 194J, 194I, etc.
- tds_provision is the provision, not the payment nature.
- tds_section is the specific Section 393 table, not the payment nature and not the provision name.

ANALYSIS RULES:
1. Evaluate the complete invoice and EVERY line item.
2. Use descriptions, HSN/SAC, vendor/customer information, PAN/GSTIN, amounts, terms and supplied evidence.
3. Use customer/payer information for payer-dependent decisions; do not confuse vendor PAN with payer PAN.
4. Do not invent missing facts.
5. Missing information is NOT zero.
6. If FY cumulative transaction history is required but not supplied, do not guess applicability; use null where necessary and set tds_needs_review=true.
7. Select a Section 393 table only when the supplied evidence supports it.
8. If the table cannot be determined reliably, use tds_section=null and set tds_needs_review=true.
9. Calculate proposed_tds_amount only when applicability, rate and base are sufficiently supported.
10. The backend independently validates your proposal and recalculates the final TDS.
11. Return ONLY the structured JSON object.'''

print('TDS prompt loaded.')

TDS prompt loaded.


In [6]:
def analyze_tds_with_groq(invoice_json: Dict[str, Any], req_id: str = "TDS-UNKNOWN") -> tuple[Dict[str, Any], str, float]:
    infer_start_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    infer_start_t = time.time()

    # Section 6: Groq Inference Status (Started)
    print("\n" + "=" * 70)
    print("🔵 GROQ TDS INFERENCE STARTED")
    print("=" * 70)
    print(f"Request ID: {req_id}")
    print(f"Model: {MODEL_NAME}")
    print(f"Start time: {infer_start_dt}")
    print(f"[{req_id}] 🔵 GROQ RUNNING")

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':'Analyze this invoice JSON and return the TDS proposal:\n\n'+json.dumps(invoice_json,ensure_ascii=False,indent=2)}
        ],
        response_format={'type':'json_schema','json_schema':{'name':'tds_assessment','schema':TDS_SCHEMA,'strict':True}},
        temperature=0,
    )
    raw_content = response.choices[0].message.content
    infer_end_t = time.time()
    infer_latency = round(infer_end_t - infer_start_t, 2)
    infer_end_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Section 6: Groq Inference Status (Completed)
    print("\n" + "=" * 70)
    print("🟢 GROQ TDS INFERENCE COMPLETED")
    print("=" * 70)
    print(f"Request ID: {req_id}")
    print(f"End time: {infer_end_dt}")
    print(f"Inference latency: {infer_latency}s")
    print(f"[{req_id}] 🟢 GROQ COMPLETED")

    result = json.loads(raw_content)
    if 'tds_assessment' not in result:
        raise ValueError('Groq response missing tds_assessment')
    return result, raw_content, infer_latency

## 5. FastAPI service & Live Status Tracking

Backend request: `POST /api/infer/tds` with `{"invoice_json": {...}}`.

Response: `{"tds_assessment": {...}}`.

In [7]:
class TDSRequest(BaseModel):
    invoice_json: Dict[str, Any]

class TDSResponse(BaseModel):
    tds_assessment: Dict[str, Any]

nest_asyncio.apply()
app = FastAPI(title='Groq TDS Proposal API')

@app.get('/health')
async def health():
    return {'status':'ok','service':'groq-tds-proposal','model':MODEL_NAME}

@app.post('/api/infer/tds', response_model=TDSResponse)
async def tds_endpoint(req: TDSRequest):
    req_id = generate_tds_request_id()
    req_start_t = time.time()
    now_dt = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    current_stage = "REQUEST_RECEIPT"

    # Section 3: Request Received
    print("\n" + "=" * 70)
    print("🟢 TDS REQUEST RECEIVED FROM BACKEND")
    print("=" * 70)
    print(f"Request ID: {req_id}")
    print(f"Timestamp: {now_dt}")
    print("HTTP Method: POST")
    print("Endpoint: /api/infer/tds")
    print("Status: RECEIVED")
    print(f"[{req_id}] 🟢 RECEIVED")

    try:
        # Section 4: Request Validation
        print("\n🟡 STATUS: VALIDATING REQUEST")
        current_stage = "VALIDATING_REQUEST"
        if not isinstance(req.invoice_json, dict) or not req.invoice_json:
            print("\n" + "=" * 70)
            print("🔴 TDS REQUEST VALIDATION FAILED")
            print("=" * 70)
            print(f"Request ID: {req_id}")
            print("Error: invoice_json must be a non-empty dictionary")
            print("Stage: VALIDATING_REQUEST")
            print("=" * 70)
            raise HTTPException(status_code=400, detail="invoice_json must be a non-empty dictionary")

        print("🟢 STATUS: REQUEST VALIDATED")
        print("\n---------------------- INVOICE JSON ----------------------")
        print(json.dumps(req.invoice_json, ensure_ascii=False, indent=2))

        # Section 5: TDS Input Preparation
        print("\n" + "=" * 70)
        print("🟡 TDS INPUT PREPARATION")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(f"Invoice number: {req.invoice_json.get('invoice_number') or 'N/A'}")
        print(f"Vendor: {req.invoice_json.get('vendor_name') or 'N/A'}")
        print(f"Customer: {req.invoice_json.get('customer_name') or 'N/A'}")
        line_items = req.invoice_json.get('line_items') or []
        print(f"Number of line items: {len(line_items)}")
        print(f"Invoice subtotal: {req.invoice_json.get('subtotal')}")
        print(f"Tax total: {req.invoice_json.get('tax_total')}")
        print(f"Total amount: {req.invoice_json.get('total_amount')}")
        print("\n🟢 STATUS: GROQ INPUT READY")

        current_stage = "INFERENCE_EXECUTION"
        async with inference_lock:
            result, raw_output, infer_latency = analyze_tds_with_groq(req.invoice_json, req_id=req_id)

        # Section 7: Raw Groq Output
        print("\n" + "=" * 70)
        print("📥 GROQ RAW OUTPUT")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(raw_output)

        # Section 8: Parse TDS Response
        print("\n🟡 STATUS: PARSING GROQ TDS RESPONSE")
        print("\n" + "=" * 70)
        print("📋 PARSED TDS ASSESSMENT")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(json.dumps(result, ensure_ascii=False, indent=2))

        # Section 9: Response Validation
        print("\n🟡 STATUS: VALIDATING TDS RESPONSE")
        current_stage = "RESPONSE_VALIDATION"
        tds_assessment = result.get("tds_assessment", {})
        if not isinstance(tds_assessment, dict):
            raise ValueError("Response missing 'tds_assessment' dictionary")

        # Structural type checks (nulls are strictly preserved)
        req_keys = ['tds_applicable', 'nature_of_payment', 'tds_provision', 'tds_section', 'tds_rate', 'tds_base_amount', 'proposed_tds_amount', 'tds_needs_review', 'tds_reasoning']
        for k in req_keys:
            if k not in tds_assessment:
                raise ValueError(f"Missing expected key in tds_assessment: {k}")
        print("🟢 STATUS: TDS RESPONSE VALIDATED")

        # Section 10: Response Sent to Backend
        current_stage = "RESPONSE_PREPARATION"
        total_latency = round(time.time() - req_start_t, 2)
        print("\n" + "=" * 70)
        print("📤 TDS RESPONSE SENT TO BACKEND")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print("\nStatus: SUCCESS")
        print(f"Inference latency: {infer_latency}s")
        print(f"Total request latency: {total_latency} seconds")
        print(f"[{req_id}] 📤 RESPONSE SENT")

        # Section 11: Final Request Summary
        print("\n" + "=" * 70)
        print("✅ TDS REQUEST COMPLETED")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print("Status: SUCCESS\n")
        print("Stages:")
        print("  ✅ Request received")
        print("  ✅ Request validated")
        print("  ✅ Input prepared")
        print("  ✅ Groq inference started")
        print("  ✅ Groq inference completed")
        print("  ✅ Output parsed")
        print("  ✅ TDS response validated")
        print("  ✅ Response sent to backend\n")
        print(f"TDS Applicable: {tds_assessment.get('tds_applicable')}")
        print(f"TDS Provision: {tds_assessment.get('tds_provision')}")
        print(f"TDS Section: {tds_assessment.get('tds_section')}")
        print(f"TDS Rate: {tds_assessment.get('tds_rate')}")
        print(f"TDS Base Amount: {tds_assessment.get('tds_base_amount')}")
        print(f"Proposed TDS: {tds_assessment.get('proposed_tds_amount')}")
        print(f"Needs Review: {tds_assessment.get('tds_needs_review')}\n")
        print(f"Groq latency: {infer_latency}s")
        print(f"Total latency: {total_latency}s")
        print("=" * 70 + "\n")

        return result

    except HTTPException:
        raise
    except Exception as exc:
        total_latency = round(time.time() - req_start_t, 2)
        # Section 12: Error Logging
        print("\n" + "=" * 70)
        print("🔴 TDS REQUEST FAILED")
        print("=" * 70)
        print(f"Request ID: {req_id}")
        print("Status: FAILED")
        print(f"Stage: {current_stage}")
        print(f"Error Type: {type(exc).__name__}")
        print(f"Error Message: {str(exc)}")
        print(f"Total Latency: {total_latency}s")
        print("\nTraceback:")
        traceback.print_exc()
        print("=" * 70 + "\n")
        raise HTTPException(status_code=500, detail=str(exc))

## 6. Local diagnostic test

In [8]:
hardware_invoice = {
    'invoice_number':'MTD/26-27/03871', 'invoice_date':'2026-08-09',
    'vendor_name':'Metro Tech Distributors Pvt. Ltd.', 'vendor_pan':'AABCM33123J',
    'customer_name':'Orion Software Labs Pvt. Ltd.', 'customer_pan':'AADCO9012R', 'customer_gstin':'07AADCO9012R1ZB',
    'subtotal':630500.0, 'tax_total':55245.0, 'total_amount':740990.0,
    'additional_fields':{'TERMS & CONDITIONS':['All hardware purchased for business/office use of the buyer.']},
    'line_items':[
      {'description':'Business Laptop - i7, 16GB RAM, 512GB SSD','hsn_code':'84713010','quantity':5,'unit_price':65000,'taxable_amount':325000},
      {'description':'24 inch LED Monitor - Full HD','hsn_code':'85285900','quantity':8,'unit_price':8500,'taxable_amount':68000},
      {'description':'Wireless Keyboard & Mouse Combo','hsn_code':'84716060','quantity':10,'unit_price':1200,'taxable_amount':12000},
      {'description':'24-Port Gigabit Network Switch','hsn_code':'85176290','quantity':3,'unit_price':15000,'taxable_amount':45000},
      {'description':'External SSD 1TB - USB 3.2','hsn_code':'84717020','quantity':15,'unit_price':6500,'taxable_amount':97500},
      {'description':'42U Server Rack Cabinet','hsn_code':'84733010','quantity':1,'unit_price':45000,'taxable_amount':45000},
      {'description':'UPS Battery Backup - 1 KVA','hsn_code':'85044090','quantity':4,'unit_price':9500,'taxable_amount':38000}
    ]
}

result, raw_text, latency = analyze_tds_with_groq(hardware_invoice, req_id="TEST-LOCAL-001")
print(json.dumps(result, indent=2, ensure_ascii=False))


🔵 GROQ TDS INFERENCE STARTED
Request ID: TEST-LOCAL-001
Model: openai/gpt-oss-120b
Start time: 2026-09-01 07:13:50
[TEST-LOCAL-001] 🔵 GROQ RUNNING

🟢 GROQ TDS INFERENCE COMPLETED
Request ID: TEST-LOCAL-001
End time: 2026-09-01 07:13:53
Inference latency: 2.53s
[TEST-LOCAL-001] 🟢 GROQ COMPLETED
{
  "tds_assessment": {
    "nature_of_payment": "Purchase of goods",
    "tds_applicable": false,
    "tds_provision": null,
    "tds_section": null,
    "tds_rate": null,
    "tds_base_amount": null,
    "proposed_tds_amount": null,
    "tds_needs_review": false,
    "tds_reasoning": "The invoice is for purchase of hardware goods. Vendor PAN (AABCM33123J) is provided, indicating that the seller's PAN is available. Under Section 393, TDS is generally required when the seller's PAN is not available or not quoted. Since the PAN is available, Section 393 does not apply, and no TDS is required."
  }
}


## 7. Start the API server & Optional ngrok tunnel

Add `NGROK_AUTH_TOKEN` to Colab Secrets if your backend needs a public URL. This is separate from the Groq API key.

In [ ]:
from google.colab import userdata
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
PORT = 8000
public_url = f"http://0.0.0.0:{PORT}"
ngrok_status = "NOT_CONFIGURED"

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(PORT)
    public_url = tunnel.public_url
    ngrok_status = "CONNECTED"

# Section 1 & 15: Server Startup Status & NGROK
print("\n" + "=" * 70)
print("🚀 TDS INFERENCE SERVER")
print("=" * 70)
print("Service: Groq TDS Proposal")
print("Status: RUNNING")
print(f"Local URL: http://0.0.0.0:{PORT}")
print(f"Health: {public_url}/health")
print("Endpoint: /api/infer/tds")
print(f"ngrok: {ngrok_status}")
print(f"Public URL: {public_url}")
print("=" * 70)

if ngrok_status == "CONNECTED":
    print("\n" + "=" * 70)
    print("🌐 TDS NGROK")
    print("=" * 70)
    print("Status: CONNECTED")
    print(f"Public URL: {public_url}")
    print(f"Health URL: {public_url}/health")
    print(f"TDS Endpoint: {public_url}/api/infer/tds")
    print("=" * 70)
    print("\n🟢 TDS SERVICE READY FOR BACKEND REQUESTS")

print("\n🟢 READY — WAITING FOR BACKEND REQUESTS\n")

config = uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning')
server = uvicorn.Server(config)
# Run continuously to accept continuous requests
await server.serve()


🚀 TDS INFERENCE SERVER
Service: Groq TDS Proposal
Status: RUNNING
Local URL: http://0.0.0.0:8000
Health: https://physiognomically-sane-dexter.ngrok-free.dev/health
Endpoint: /api/infer/tds
ngrok: CONNECTED
Public URL: https://physiognomically-sane-dexter.ngrok-free.dev

🌐 TDS NGROK
Status: CONNECTED
Public URL: https://physiognomically-sane-dexter.ngrok-free.dev
Health URL: https://physiognomically-sane-dexter.ngrok-free.dev/health
TDS Endpoint: https://physiognomically-sane-dexter.ngrok-free.dev/api/infer/tds

🟢 TDS SERVICE READY FOR BACKEND REQUESTS

🟢 READY — WAITING FOR BACKEND REQUESTS



## 8. Backend contract

Every request is logged with full lifecycle diagnostics.

The backend independently runs its own TDS validator and decides the final result. This notebook never writes the final TDS decision.